# 高级模型优化技术教程

本教程涵盖高级模型优化技术：
1. 混合精度量化
2. SmoothQuant
3. Movement Pruning
4. 自蒸馏
5. KV Cache 优化
6. Speculative Decoding

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## 1. 混合精度量化

不同层使用不同精度，平衡精度和效率。

In [ ]:
class LayerSensitivityAnalyzer:
    """分析每层对量化的敏感度"""
    def __init__(self, model, calibration_data):
        self.model = model
        self.calibration_data = calibration_data
        self.sensitivity = {}
    
    def analyze(self):
        self.model.eval()
        with torch.no_grad():
            baseline_output = self.model(self.calibration_data)
        
        for name, module in self.model.named_modules():
            if isinstance(module, (nn.Linear, nn.Conv2d)):
                original_weight = module.weight.data.clone()
                
                # 模拟 INT8 量化
                scale = module.weight.abs().max() / 127
                quantized = torch.round(module.weight / scale) * scale
                module.weight.data = quantized
                
                with torch.no_grad():
                    quant_output = self.model(self.calibration_data)
                
                # 计算输出差异
                mse = F.mse_loss(quant_output, baseline_output).item()
                self.sensitivity[name] = mse
                
                module.weight.data = original_weight
        
        return self.sensitivity

# 示例模型
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(64, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, 10)
    
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

model = SimpleNet()
calibration_data = torch.randn(32, 64)

analyzer = LayerSensitivityAnalyzer(model, calibration_data)
sensitivity = analyzer.analyze()

print('Layer Sensitivity Analysis:')
for name, sens in sensitivity.items():
    print(f'  {name}: {sens:.6f}')

## 2. SmoothQuant

平滑激活值异常值，将量化难度从激活值转移到权重。

In [ ]:
def smooth_quant(weight, activation_scale, alpha=0.5):
    """
    SmoothQuant 实现
    
    W' = W * diag(s)
    X' = X * diag(s)^(-1)
    """
    act_max = activation_scale.abs()
    weight_max = weight.abs().max(dim=0)[0]
    
    smooth_factor = (act_max.pow(alpha) / (weight_max.pow(1 - alpha) + 1e-5)).clamp(min=1e-5)
    smoothed_weight = weight * smooth_factor.unsqueeze(0)
    
    return smoothed_weight, smooth_factor

# 模拟有异常值的激活
weight = torch.randn(128, 64)
activation_scale = torch.randn(64).abs()
activation_scale[10] = 100.0  # 异常值

smoothed_weight, smooth_factor = smooth_quant(weight, activation_scale)

print(f'Original weight range: [{weight.min():.3f}, {weight.max():.3f}]')
print(f'Smoothed weight range: [{smoothed_weight.min():.3f}, {smoothed_weight.max():.3f}]')
print(f'Smooth factor range: [{smooth_factor.min():.3f}, {smooth_factor.max():.3f}]')

## 3. Movement Pruning

基于训练过程中权重变化方向进行剪枝。

In [ ]:
class MovementPruner:
    def __init__(self, model):
        self.model = model
        self.scores = {}
        for name, param in model.named_parameters():
            if 'weight' in name:
                self.scores[name] = torch.zeros_like(param)
    
    def update_scores(self):
        for name, param in self.model.named_parameters():
            if name in self.scores and param.grad is not None:
                movement = param.data * param.grad
                self.scores[name] += movement
    
    def get_mask(self, sparsity):
        masks = {}
        for name, score in self.scores.items():
            threshold = torch.quantile(score.flatten(), sparsity)
            masks[name] = (score > threshold).float()
        return masks

# 模拟训练
model = SimpleNet()
pruner = MovementPruner(model)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

for _ in range(10):
    x = torch.randn(32, 64)
    y = torch.randint(0, 10, (32,))
    
    optimizer.zero_grad()
    output = model(x)
    loss = F.cross_entropy(output, y)
    loss.backward()
    
    pruner.update_scores()
    optimizer.step()

masks = pruner.get_mask(sparsity=0.3)
for name, mask in masks.items():
    sparsity = 1 - mask.mean().item()
    print(f'{name}: sparsity = {sparsity:.2%}')

## 4. 自蒸馏 (Self-Distillation)

模型自己作为教师，深层指导浅层。

In [ ]:
class SelfDistillationNet(nn.Module):
    def __init__(self, input_dim=64, hidden_dim=128, num_classes=10):
        super().__init__()
        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.layer2 = nn.Linear(hidden_dim, hidden_dim)
        self.layer3 = nn.Linear(hidden_dim, hidden_dim)
        self.head = nn.Linear(hidden_dim, num_classes)
        
        # 早期退出分类器
        self.exit1 = nn.Linear(hidden_dim, num_classes)
        self.exit2 = nn.Linear(hidden_dim, num_classes)
    
    def forward(self, x, return_exits=False):
        h1 = F.relu(self.layer1(x))
        h2 = F.relu(self.layer2(h1))
        h3 = F.relu(self.layer3(h2))
        final = self.head(h3)
        
        if return_exits:
            exit1 = self.exit1(h1)
            exit2 = self.exit2(h2)
            return final, [exit1, exit2]
        return final

def self_distillation_loss(final_output, exit_outputs, labels, temperature=3.0):
    hard_loss = F.cross_entropy(final_output, labels)
    
    soft_teacher = F.softmax(final_output.detach() / temperature, dim=-1)
    distill_loss = 0
    for exit_out in exit_outputs:
        soft_student = F.log_softmax(exit_out / temperature, dim=-1)
        distill_loss += F.kl_div(soft_student, soft_teacher, reduction='batchmean')
    
    return hard_loss + 0.5 * distill_loss * (temperature ** 2)

# 训练示例
model = SelfDistillationNet()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(5):
    x = torch.randn(32, 64)
    y = torch.randint(0, 10, (32,))
    
    final, exits = model(x, return_exits=True)
    loss = self_distillation_loss(final, exits, y)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    print(f'Epoch {epoch+1}: Loss = {loss.item():.4f}')

## 5. KV Cache 优化

优化 Transformer 自回归推理。

In [ ]:
class KVCache:
    def __init__(self, batch_size, max_length, num_heads, head_dim):
        self.k_cache = torch.zeros(batch_size, num_heads, max_length, head_dim)
        self.v_cache = torch.zeros(batch_size, num_heads, max_length, head_dim)
        self.current_length = 0
    
    def update(self, new_k, new_v):
        seq_len = new_k.size(2)
        self.k_cache[:, :, self.current_length:self.current_length+seq_len] = new_k
        self.v_cache[:, :, self.current_length:self.current_length+seq_len] = new_v
        self.current_length += seq_len
    
    def get(self):
        return self.k_cache[:, :, :self.current_length], self.v_cache[:, :, :self.current_length]

class CachedAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
    
    def forward(self, x, kv_cache=None):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        
        if kv_cache is not None:
            kv_cache.update(k, v)
            k, v = kv_cache.get()
        
        attn = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn = F.softmax(attn, dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, T, C)
        return self.out_proj(out)

# 测试 KV Cache
attn = CachedAttention(d_model=64, num_heads=4)
cache = KVCache(batch_size=1, max_length=100, num_heads=4, head_dim=16)

# 模拟自回归生成
for i in range(5):
    x = torch.randn(1, 1, 64)  # 每次一个 token
    out = attn(x, kv_cache=cache)
    print(f'Step {i+1}: Cache length = {cache.current_length}')

## 6. Speculative Decoding

使用小模型加速大模型推理。

In [ ]:
class SimpleLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([nn.Linear(d_model, d_model) for _ in range(n_layers)])
        self.head = nn.Linear(d_model, vocab_size)
    
    def forward(self, x):
        h = self.embed(x)
        for layer in self.layers:
            h = F.relu(layer(h))
        return self.head(h)

class SpeculativeDecoder:
    def __init__(self, draft_model, target_model, k=4):
        self.draft = draft_model
        self.target = target_model
        self.k = k
    
    @torch.no_grad()
    def generate_step(self, input_ids):
        # Draft model 生成 k 个 token
        draft_tokens = []
        draft_probs = []
        current = input_ids.clone()
        
        for _ in range(self.k):
            logits = self.draft(current)[:, -1]
            probs = F.softmax(logits, dim=-1)
            token = torch.argmax(probs, dim=-1, keepdim=True)
            draft_tokens.append(token)
            draft_probs.append(probs.gather(-1, token))
            current = torch.cat([current, token], dim=1)
        
        # Target model 验证
        target_logits = self.target(current)
        target_probs = F.softmax(target_logits[:, -self.k-1:-1], dim=-1)
        
        # 接受/拒绝
        accepted = 0
        for i, (token, draft_p) in enumerate(zip(draft_tokens, draft_probs)):
            target_p = target_probs[:, i].gather(-1, token)
            if (target_p >= draft_p).all():
                accepted += 1
            else:
                break
        
        return current[:, :input_ids.size(1) + accepted + 1], accepted

# 测试
vocab_size = 100
draft = SimpleLM(vocab_size, d_model=32, n_layers=2)
target = SimpleLM(vocab_size, d_model=64, n_layers=4)

decoder = SpeculativeDecoder(draft, target, k=4)
input_ids = torch.randint(0, vocab_size, (1, 5))

output, accepted = decoder.generate_step(input_ids)
print(f'Input length: {input_ids.size(1)}')
print(f'Output length: {output.size(1)}')
print(f'Accepted tokens: {accepted}')

## 总结

| 技术 | 适用场景 | 优势 |
|:-----|:---------|:-----|
| 混合精度量化 | 精度敏感模型 | 平衡精度和效率 |
| SmoothQuant | LLM 量化 | 处理激活异常值 |
| Movement Pruning | 微调剪枝 | 保留重要权重 |
| 自蒸馏 | 模型压缩 | 无需额外教师 |
| KV Cache | Transformer 推理 | 避免重复计算 |
| Speculative Decoding | LLM 推理 | 加速自回归生成 |